# Statistical Test for Price with other Variables 

In earlier EDA, we have seen many factors can contribute to cosmetic products' price. Here we will use statistical test to evaluate the significance of these variables. We will look at:

* Product Category
* Brand
* Ingredient
    * Number of ingredient
    * Ingredient rating
    * Ingredient category
    * Individule ingredient

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

import sys
sys.path.append('../utilities')
from data_utils import *
from stats_utils import *

Read data and clean a little bit.

In [ ]:
# read cleaned data
skin_care_df = pd.read_csv('../data_cleaning/skin_care_cleaned.csv')
body_care_df = pd.read_csv('../data_cleaning/body_care_cleaned.csv')
makeup_df = pd.read_csv('../data_cleaning/makeup_cleaned.csv')

# merge skin-care, body-care and makeup table
merged_df = pd.concat([skin_care_df,body_care_df,makeup_df], sort=False).reset_index(drop=True)

# rename some columns so the formular for statsmodel can work
for col in merged_df.columns.values:
    col_new = col.replace(' ','_').replace('-','_').replace(':','').replace('/','_')
    merged_df.rename({col : col_new}, axis=1, inplace=True)
    skin_care_df.rename({col : col_new}, axis=1, inplace=True)
    body_care_df.rename({col : col_new}, axis=1, inplace=True)
    makeup_df.rename({col : col_new}, axis=1, inplace=True)

# drop outlier
skin_care_df = skin_care_df.loc[merged_df['price']<1000]
merged_df = merged_df.loc[merged_df['price']<1000]

### ANOVA test for product category

In [ ]:
model = ols('price ~ product_category', data=merged_df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

The large F-statistic and small p-value tells us at least one product category has different price distribution with the others, but it didn't tell us which category. So to get more information, we can try to do pair-wise t-test.

Do pairwise t-test to compare group means for the six largest categories.

In [ ]:
def pairwise_t(df, groups, x, y):
    
    ts = pd.DataFrame(index=groups, columns=groups)
    p_values = pd.DataFrame(index=groups, columns=groups)
    
    for i in groups:
        values_i = df[y].loc[df[x]==i]
        for j in groups:
            values_j = df[y].loc[df[x]==j]
            t, p_value = stats.ttest_ind(values_i, values_j)
            ts.loc[i,j] = t
            p_values.loc[i,j] = p_value
    return ts.astype('float'), p_values.astype('float')

categories = ['Nighttime Moisturizer', 'Cleansers', 'Serum', 'Exfoliants', 'Lip Stick/Gloss/Liner', 'Eye Cream & Treatment']
ts, p_values = pairwise_t(merged_df, categories, 'product_category', 'price')
p_values

So from the p-values, we see that it is hard to distinguish the price of nighttime moisturizers and eye cream & treatment, the second hard pair would be cleansers and lip stick/gloss/liner. The other categories are distinguishable. Also visualize the p-values using heatmaps, with the sign of t-values encoded so that red block indicates the row category is more expensive than the column category, blue indicates that row category is cheaper than the column category. Also, darker color means the two brand is easy to tell apart (distinguishable). From the heatmap, it is clear to see that serums are the most expensive among the six categories and lip products are the cheapest.

In [ ]:
ax=sns.heatmap(ts.apply(np.sign) * p_values.apply(np.log), cmap="RdBu")
ax.set_aspect("equal")
plt.show()

### ANOVA test for brand

In [ ]:
skin_care_df = skin_care_df.drop(['product_category'],axis=1).drop_duplicates()
body_care_df = body_care_df.drop(['product_category'],axis=1).drop_duplicates()
makeup_df = makeup_df.drop(['product_category'],axis=1).drop_duplicates()
df = merged_df.drop(['product_category'],axis=1).drop_duplicates()

In [ ]:
model = ols('price ~ brand', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

As we expected, price also varies with brand. Again, we can do pairwise t-test on the six largest brand (large means having more product in the dataset).

In [ ]:
brands = ["Neutrogena", "Clinique", "L'Oreal Paris", "Paula's Choice Skincare", "Clarins", "Shiseido"]
ts, p_values = pairwise_t(df, brands, 'brand', 'price')

ax=sns.heatmap(ts.apply(np.sign) * p_values.apply(np.log), cmap="RdBu")
ax.set_aspect("equal")
plt.show()

So, we find Clarins and Shiseido are the most expensive brands of the six, and hard to distinguish among themselves, Neutrogena is the cheapest.

### Tests For ingredient

drop products without ingredient information.

In [ ]:
skin_care_df = skin_care_df.loc[~skin_care_df['ingredient'].isnull()]
body_care_df = body_care_df.loc[~body_care_df['ingredient'].isnull()]
makeup_df = makeup_df.loc[~makeup_df['ingredient'].isnull()]
df = df.loc[~df['ingredient'].isnull()]

#### Slope test for number of ingredients

In the EDA we have done earlier, we found price seem to be positively correlated with number of ingredient, and the effect is stronger in skin care products. We will examine the statistic significance using slope tests on the number of ingredient.

In [ ]:
n_ingredient_lr = lr_stats(skin_care_df, ['n_inactive_ingredient', 'n_active_ingredient'], 'price')
n_ingredient_lr

In [ ]:
n_ingredient_lr = lr_stats(body_care_df, ['n_inactive_ingredient', 'n_active_ingredient'], 'price')
n_ingredient_lr

In [ ]:
n_ingredient_lr = lr_stats(makeup_df, ['n_inactive_ingredient', 'n_active_ingredient'], 'price')
n_ingredient_lr

Surperisingly, we find that even in body care and makeup products, the number of ingredients have statistical significant with price, although the p-values are larger compare to that in skin care products. It could be the case of small practical significance but large statistical significance. The r-values are small, so the underlining correlation may be buried under large variance. Also, the number of active ingredient in skin care is negatively correlated with price, the reason might be that the products containing active ingredient such as sunscreens are cheaper categories.

#### Slope test for ingredient rating

From the statistic test, we see that the rating for inactive ingredients do has statistical significance with price, which is not obvious on the rating-price scatter plot we have in EDA earlier. Again, it could be the case of small practical significance but large statistical significance, the slopes are large (meaning the fitted lines are almost vertical), and for makeup and body care, the slopes are negative.

In [ ]:
ingredient_rating_lr = lr_stats(skin_care_df, 
      ['active_mean_rating', 'inactive_mean_rating', 'inactive_mean_rating_w1',
       'inactive_mean_rating_w2'], 'price')
ingredient_rating_lr

In [ ]:
ingredient_rating_lr = lr_stats(body_care_df, 
      ['active_mean_rating', 'inactive_mean_rating', 'inactive_mean_rating_w1',
       'inactive_mean_rating_w2'], 'price')
ingredient_rating_lr

In [ ]:
ingredient_rating_lr = lr_stats(makeup_df, 
      ['active_mean_rating', 'inactive_mean_rating', 'inactive_mean_rating_w1',
       'inactive_mean_rating_w2'], 'price')
ingredient_rating_lr

#### F-test for ingredient category count

We will use statsmodel to perform linear regression on ingredient category count features. The model summary reports a large F-statistic, which indicates at least one of the features should have none zeros slope. It also report the t-statitic for each feature, which can help us identify possible features that have significant influence on price.

In [ ]:
inactive_cat_count = [f_ for f_ in df.columns.values if f_.find('inactive_cat_count')>=0]
fitting_formula = "price ~ " + " + ".join(inactive_cat_count)
model = ols(fitting_formula, data=df).fit()
model.summary()

So we find the top 5 ingredient categories that have largest (absolute) t-statistic/smallest p-values are:
* Skin Restoring
* Antioxident
* Texture_Enhancer
* Plant Extract
* Fragrance

These match the categories we find in early EDA that have large correlation with price.

#### chi-square test for individule ingredient

We will use chi-square chi square homogeneity test on some "interesting" ingredients --- ingredients that appear in a considerable number of products but are not too common. We do see chi-square test identify a few ingredients which cosmetic companies advertise a lot, such as piptides, retinol, xxx extract...

In [ ]:
ingredient_count_df = get_matching_ingredient_count(df)

interesting_ingredients = []
for col in ingredient_count_df.columns.values:
    count_ingredient = ingredient_count_df[col].sum()
    if count_ingredient > 100 and count_ingredient < 500:
       interesting_ingredients.append(col) 
    
price_band, bins = pd.qcut(df['price'], q=[0, 0.2, 0.4, 0.6, 0.8, 1], labels=False, retbins=True)
ingredient_count_df['price_band'] = price_band.reset_index(drop=True)

results = chi2_contingency(ingredient_count_df, interesting_ingredients, 'price_band').sort_values('p-value')
results.head(50)

We have identified quite a number of ingredients with pretty small p-values. However, we should keep in mind that when the features are colinear, adding them all together to the machine learning model may not be very helpful because there aren't much new information.